# 01. VGGFace2 목록 준비
같은 목록과 label mapping을 모든 seed/head가 공유합니다.

## 1. 설정

In [ ]:
from pathlib import Path
import sys, json
BASE_DIR = Path("F:/FaceX-Zoo")
SETTINGS_PATH = BASE_DIR / "fine-tune/data/notebook_settings.json"
for path in (BASE_DIR, BASE_DIR / "fine-tune"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from modules.notebook_workflow import resolve_settings, save_settings, plan_matrix
# 00의 공통값을 읽습니다. 아래 변수 변경이 저장값/YAML보다 우선합니다.
saved = resolve_settings(BASE_DIR, SETTINGS_PATH, require_saved=True)
SEEDS = list(saved["project"]["seed_list"])  # 예: [8972]. 이후 단계도 같은 목록을 사용하세요.
HEADS = list(saved["campaign"]["heads"])  # 기본 세 head. 02는 ArcFace 부모만 생성합니다.
KEEP_RAW_RESULTS = saved["project"]["keep_raw_results"]
NUM_WORKERS = 0
DEVICE = saved["project"]["device"]
DATASET = dict(saved["dataset"])  # 경로·input_size·pixel_mean/std
BACKBONE = dict(saved["backbone"])
TRAINING = dict(saved["training"])  # epoch·batch·학습률 등
HEAD_PARAMETERS = saved["heads"]
PATHS = dict(saved["paths"])
REUSE_COMPLETED = saved["campaign"]["reuse_completed"]
EXECUTE = False  # True일 때만 목록 생성/학습/export를 수행합니다.
VAL_EVERY = 20  # 매 20번째 이미지를 validation으로 분리 (자체 선택)
# 현재 노트북 변수 -> 실제 모듈 설정. 입력 112x112 이외는 현 구조에서 명시적 중단.
DATASET.update(train_archive=str(Path(DATASET["raw_root"])/"data/vggface2_train.tar.gz"),
               test_archive=str(Path(DATASET["raw_root"])/"data/vggface2_test.tar.gz"),
               meta_csv=str(Path(DATASET["raw_root"])/"meta/identity_meta.csv"))
settings = resolve_settings(BASE_DIR, SETTINGS_PATH, overrides={
    "project": {"seed": SEEDS[0], "seed_list": SEEDS, "keep_raw_results": KEEP_RAW_RESULTS, "device": DEVICE},
    "campaign": {"heads": HEADS, "reuse_completed": REUSE_COMPLETED},
    "dataset": DATASET, "backbone": BACKBONE, "heads": HEAD_PARAMETERS,
    "training": {**TRAINING, "num_workers": NUM_WORKERS}, "paths": PATHS,
})


## 2. 경로 확인

In [ ]:
print(DATASET)
print("EXECUTE:", EXECUTE, "VAL_EVERY:", VAL_EVERY)

## 3. 목록 생성 또는 같은 목록 재사용

In [ ]:
from modules.notebook_workflow import prepare_indices
split_result = prepare_indices(settings, VAL_EVERY) if EXECUTE else None
print(split_result if EXECUTE else "계획 모드: 목록 생성 미실행")

## 4. 실제 이미지 전처리 확인

In [ ]:
from modules.notebook_workflow import make_loaders
if EXECUTE:
    train_loader, val_loader = make_loaders(settings)
    images, labels = next(iter(val_loader))
    print(tuple(images.shape), float(images.min()), float(images.max()))
    print("workers:", train_loader.num_workers, val_loader.num_workers)
    del train_loader, val_loader
else:
    print("계획 모드: 이미지 읽기 미실행")

## 5. 결과 해석

In [ ]:
print("이미지 단위 분할이며 identity-disjoint 검증이 아닙니다.")
print("기존 목록과 다르면 덮어쓰지 않습니다. 새 목록 경로를 지정하세요.")
print("표본 존재/읽기 확인은 전체 데이터 무결성 확인이 아닙니다.")